# How this machine drives the GPU machine (the `.sh` files, in Python)

This notebook re-tells `finetune/run_finetune.sh` and `finetune/setup_vm.sh` in Python, step by step,
so you can see what each one does. **It is a dry run:** every cell *prints* the exact command it would
send instead of running it. Nothing touches a VM and nothing costs money.

Two machines are involved:
- **This EC2 machine (T4)** holds the photos and the code, and decides what to do.
- **The Nebius H100 VM** only does the heavy part: training. It has no project code of its own.

They talk through two standard tools:
- **`ssh`** runs a command *on the other machine*: `ssh ubuntu@<IP> "nvidia-smi"` shows the VM's GPU.
- **`rsync`** copies files between them, and skips files that are already there.

To really run a step, set `DRY_RUN = False` and put the VM's IP in `VM_IP` (only with a VM running!).

In [1]:
import shlex, subprocess

VM_IP = "1.2.3.4"          # the VM's public IP from the Nebius console (changes every time a VM is created)
VM_USER = "ubuntu"
VM_DIR = "selfheal"        # working folder on the VM, under /home/ubuntu
DRY_RUN = True             # True: print commands only. False: actually run them.

def run(cmd: str) -> None:
    """Run a shell command on THIS machine (or just print it)."""
    print("$", cmd)
    if not DRY_RUN:
        subprocess.run(cmd, shell=True, check=True)

def vm(command: str) -> None:
    """Run `command` ON THE VM, inside its work folder. This is run_finetune.sh's `vm()`.
    BatchMode=yes: never stop to ask for a password; fail instead (we use the SSH key)."""
    run(f"ssh -o BatchMode=yes {VM_USER}@{VM_IP} " + shlex.quote(f"cd ~/{VM_DIR} && {command}"))

def to_vm(*local_paths: str, dest: str = "") -> None:
    """Copy files/folders from here to the VM (rsync -a keeps folders and skips unchanged files)."""
    run(f"rsync -a {' '.join(local_paths)} {VM_USER}@{VM_IP}:~/{VM_DIR}/{dest}")

def from_vm(remote: str, local: str) -> None:
    """Copy files from the VM back here."""
    run(f"rsync -a {VM_USER}@{VM_IP}:~/{VM_DIR}/{remote} {local}")

print("helpers ready; DRY_RUN =", DRY_RUN)

helpers ready; DRY_RUN = True


## Step 0 — a safety switch first (cost control)

The very first thing sent to a new VM: *switch yourself off in N minutes, whatever happens.* If a job hangs
or I lose the connection, the bill stops anyway. `sudo shutdown -h +300` = power off in 300 minutes.

In [2]:
SHUTDOWN_MIN = 300
run(f"ssh -o BatchMode=yes {VM_USER}@{VM_IP} 'sudo shutdown -h +{SHUTDOWN_MIN}'")

$ ssh -o BatchMode=yes ubuntu@1.2.3.4 'sudo shutdown -h +300'


## Step 1 — `data`: build the training examples (on this machine)

Each rendered scene becomes one line of JSON: the inspector's question, the photo paths, and the
correct answer. Python scripts do this here, no VM needed. (Tonight's mixed set was built the same way
with `build_sorting_dataset.py`.)

In [3]:
run("python finetune/build_dataset.py runs/scenes/ftrain_v1 --config config/selfheal_fastener.yaml --out finetune/data/train.jsonl")
run("python finetune/build_sorting_dataset.py runs/sorting/train_data --out finetune/data/sort_train.jsonl")

$ python finetune/build_dataset.py runs/scenes/ftrain_v1 --config config/selfheal_fastener.yaml --out finetune/data/train.jsonl
$ python finetune/build_sorting_dataset.py runs/sorting/train_data --out finetune/data/sort_train.jsonl


## Step 2 — `push`: copy what the VM needs

Only three scripts, the example files, and the photos. The VM never gets the rest of the project.
`--include '*.png' --exclude '*'` = copy only the pictures, not the answer-key files (the answers are
already inside the JSONL).

In [4]:
vm("mkdir -p scenes")
to_vm("finetune/setup_vm.sh", "finetune/train_lora.py", "finetune/generate.py",
      "finetune/data/mix_train.jsonl", "finetune/data/dev.jsonl", "finetune/data/eval.jsonl")
for folder in ["runs/scenes/ftrain_v1", "runs/scenes/fdev_v2", "runs/scenes/feval_v1",
               "runs/sorting/train_data", "runs/sorting/dev_data"]:
    run(f"rsync -a --include='*/' --include='*.png' --exclude='*' {folder} {VM_USER}@{VM_IP}:~/{VM_DIR}/scenes/")

$ ssh -o BatchMode=yes ubuntu@1.2.3.4 'cd ~/selfheal && mkdir -p scenes'
$ rsync -a finetune/setup_vm.sh finetune/train_lora.py finetune/generate.py finetune/data/mix_train.jsonl finetune/data/dev.jsonl finetune/data/eval.jsonl ubuntu@1.2.3.4:~/selfheal/
$ rsync -a --include='*/' --include='*.png' --exclude='*' runs/scenes/ftrain_v1 ubuntu@1.2.3.4:~/selfheal/scenes/
$ rsync -a --include='*/' --include='*.png' --exclude='*' runs/scenes/fdev_v2 ubuntu@1.2.3.4:~/selfheal/scenes/
$ rsync -a --include='*/' --include='*.png' --exclude='*' runs/scenes/feval_v1 ubuntu@1.2.3.4:~/selfheal/scenes/
$ rsync -a --include='*/' --include='*.png' --exclude='*' runs/sorting/train_data ubuntu@1.2.3.4:~/selfheal/scenes/
$ rsync -a --include='*/' --include='*.png' --exclude='*' runs/sorting/dev_data ubuntu@1.2.3.4:~/selfheal/scenes/


## Step 3 — `setup`: what `setup_vm.sh` does on the VM

A fresh Nebius GPU image has drivers and CUDA but no Python packages. `setup_vm.sh`:
1. installs **uv** (a fast Python package installer);
2. makes a private Python environment `.venv`;
3. installs **PyTorch** (built for CUDA 12.4), **transformers** (loads the model), **peft** (LoRA),
   **accelerate**, **qwen-vl-utils** (image handling for this model family);
4. downloads **nvidia/Cosmos-Reason1-7B** from Hugging Face (~16 GB);
5. prints a check that the GPU is visible.

Running it twice is safe: anything already there is skipped. Below are its real commands.

In [5]:
setup_vm_sh = [
    "curl -LsSf https://astral.sh/uv/install.sh | sh",
    "uv venv --python 3.10 .venv",
    "uv pip install --python .venv/bin/python --index-url https://download.pytorch.org/whl/cu124 "
    "--extra-index-url https://pypi.org/simple torch==2.5.1 torchvision==0.20.1",
    "uv pip install --python .venv/bin/python 'transformers>=4.51,<4.56' 'peft>=0.13' 'accelerate>=1.0' "
    "'qwen-vl-utils>=0.0.8' pillow pydantic huggingface_hub hf_transfer",
    ".venv/bin/python -c \"from huggingface_hub import snapshot_download; snapshot_download('nvidia/Cosmos-Reason1-7B')\"",
]
for line in setup_vm_sh:
    print("   on the VM:", line)
print()
vm("bash setup_vm.sh")   # what run_finetune.sh actually sends

   on the VM: curl -LsSf https://astral.sh/uv/install.sh | sh
   on the VM: uv venv --python 3.10 .venv
   on the VM: uv pip install --python .venv/bin/python --index-url https://download.pytorch.org/whl/cu124 --extra-index-url https://pypi.org/simple torch==2.5.1 torchvision==0.20.1
   on the VM: uv pip install --python .venv/bin/python 'transformers>=4.51,<4.56' 'peft>=0.13' 'accelerate>=1.0' 'qwen-vl-utils>=0.0.8' pillow pydantic huggingface_hub hf_transfer
   on the VM: .venv/bin/python -c "from huggingface_hub import snapshot_download; snapshot_download('nvidia/Cosmos-Reason1-7B')"

$ ssh -o BatchMode=yes ubuntu@1.2.3.4 'cd ~/selfheal && bash setup_vm.sh'


## Step 4 — `train` / `retrain`: start training, detached

`nohup … &` = start it and let it keep running even if the SSH connection drops (a dropped connection
would otherwise kill a 2-hour job). The output goes to a log file on the VM.

Tonight's retrain starts two copies side by side:
- **A**: LoRA on the language model **and** the vision encoder, rank 16, lr 1e-4 (the Oct 12 recipe);
- **B**: `--freeze-vision`, rank 8, lr 5e-5 (gentler).

In [6]:
common = "--data mix_train.jsonl --val val_mix.jsonl --images scenes --epochs 1 --save-every 1200"
vm(f"nohup .venv/bin/python train_lora.py {common} --out adapter_A > train_A.log 2>&1 &")
vm(f"nohup .venv/bin/python train_lora.py {common} --out adapter_B --freeze-vision --rank 8 --lr 5e-5 > train_B.log 2>&1 &")

$ ssh -o BatchMode=yes ubuntu@1.2.3.4 'cd ~/selfheal && nohup .venv/bin/python train_lora.py --data mix_train.jsonl --val val_mix.jsonl --images scenes --epochs 1 --save-every 1200 --out adapter_A > train_A.log 2>&1 &'
$ ssh -o BatchMode=yes ubuntu@1.2.3.4 'cd ~/selfheal && nohup .venv/bin/python train_lora.py --data mix_train.jsonl --val val_mix.jsonl --images scenes --epochs 1 --save-every 1200 --out adapter_B --freeze-vision --rank 8 --lr 5e-5 > train_B.log 2>&1 &'


## Step 5 — watch it (without paying for idle time)

Every minute, this machine copies the two log files back (`runs/retrain_logs/sync_vm_logs.sh`), and
`scripts/watchdog.py` reads them: it raises an alarm on an error, on 10 minutes without progress, or
when training is done. Reading a log by hand looks like this:

In [7]:
vm("tail -n 2 train_A.log train_B.log")
run("python scripts/watchdog.py --log runs/retrain_logs/train_A.log --process sync_vm_logs.sh "
    "--progress '\\[train\\] \\d+/3600' --done '\\[train\\] DONE' --stall-min 10")

$ ssh -o BatchMode=yes ubuntu@1.2.3.4 'cd ~/selfheal && tail -n 2 train_A.log train_B.log'
$ python scripts/watchdog.py --log runs/retrain_logs/train_A.log --process sync_vm_logs.sh --progress '\[train\] \d+/3600' --done '\[train\] DONE' --stall-min 10


## Step 6 — `pull`: bring the results home, then delete the VM

The trained adapters (~100-200 MB each) and the logs come back. After that, nothing on the VM is needed:
**delete it in the Nebius console** (a stopped VM still pays for its disk).

In [8]:
from_vm("adapter_A/", "finetune/adapter_A/")
from_vm("adapter_B/", "finetune/adapter_B/")
from_vm("train_*.log", "runs/retrain_logs/")

$ rsync -a ubuntu@1.2.3.4:~/selfheal/adapter_A/ finetune/adapter_A/
$ rsync -a ubuntu@1.2.3.4:~/selfheal/adapter_B/ finetune/adapter_B/
$ rsync -a ubuntu@1.2.3.4:~/selfheal/train_*.log runs/retrain_logs/


## Step 7 — `score`: test them on this machine (free)

The adapters run on the T4 in 4-bit and are scored with the same scorer as every other model.

In [9]:
run("~/cosmos_t4/.venv/bin/python finetune/generate.py --data finetune/data/eval.jsonl --images runs/scenes "
    "--adapter finetune/adapter_B --quant 4bit --out finetune/data/gen/gen_B_eval.jsonl")
run("python finetune/score_generations.py finetune/data/eval.jsonl finetune/data/gen/gen_B_eval.jsonl "
    "--scene-dir runs/scenes/feval_v1 --model 'variant B' --out runs/bench/feval_v1__retrain_B")

$ ~/cosmos_t4/.venv/bin/python finetune/generate.py --data finetune/data/eval.jsonl --images runs/scenes --adapter finetune/adapter_B --quant 4bit --out finetune/data/gen/gen_B_eval.jsonl
$ python finetune/score_generations.py finetune/data/eval.jsonl finetune/data/gen/gen_B_eval.jsonl --scene-dir runs/scenes/feval_v1 --model 'variant B' --out runs/bench/feval_v1__retrain_B


## Where the real files are

| This notebook's step | Real file |
|---|---|
| helpers `vm()`, `to_vm()` | top of `finetune/run_finetune.sh` |
| steps 1-2, 4, 6-7 | `step_data`, `step_push`, `step_train`/`step_retrain`, `step_pull`, `step_score` in `finetune/run_finetune.sh` |
| step 3 | `finetune/setup_vm.sh` |
| the training itself | `finetune/train_lora.py` (Python, runs on the VM) |
| watching | `scripts/watchdog.py`, `runs/retrain_logs/sync_vm_logs.sh` |

Run a real step from a terminal, e.g. `VM_IP=1.2.3.4 ./finetune/run_finetune.sh push`.